# 02 · Preprocessing
Chronological 70/30 train/test split, then (fit on train only): drop >50%-missing and constant sensors, median-impute, clip to 0.5–99.5% quantiles, drop highly correlated sensors.
Writes `data/processed/train.csv` and `data/processed/test.csv`.

In [1]:
import pandas as pd
import numpy as np

X = pd.read_csv("../data/secom.data", sep=r"\s+", header=None)
labels = pd.read_csv("../data/secom_labels.data", sep=r"\s+", header=None,
                     names=["label", "timestamp"])
labels["timestamp"] = pd.to_datetime(labels["timestamp"], dayfirst=True)
y = (labels["label"] == 1).astype(int)

# Sort everything chronologically, keeping rows aligned
order = labels["timestamp"].argsort(kind="stable")
X = X.iloc[order].reset_index(drop=True)
y = y.iloc[order].reset_index(drop=True)
ts = labels["timestamp"].iloc[order].reset_index(drop=True)

X.columns = [f"s{i}" for i in X.columns]   # name sensors s0 ... s589
print(X.shape, ts.min(), "to", ts.max())

(1567, 590) 2008-07-19 11:55:00 to 2008-10-17 06:07:00


In [2]:
split = int(len(X) * 0.7)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]
ts_train, ts_test = ts.iloc[:split], ts.iloc[split:]

print(f"Train: {len(X_train)} wafers, {y_train.sum()} fails ({y_train.mean():.1%})")
print(f"Test:  {len(X_test)} wafers, {y_test.sum()} fails ({y_test.mean():.1%})")
print("Split date:", ts_test.iloc[0])

Train: 1096 wafers, 78 fails (7.1%)
Test:  471 wafers, 26 fails (5.5%)
Split date: 2008-09-26 03:12:00


In [3]:
missing_pct = X_train.isna().mean()
drop_missing = missing_pct[missing_pct > 0.5].index
print("Dropping (>50% missing):", len(drop_missing))

Dropping (>50% missing): 24


In [4]:
nunique = X_train.nunique(dropna=True)
drop_constant = nunique[nunique <= 1].index
print("Dropping (constant):", len(drop_constant))

drop_cols = drop_missing.union(drop_constant)
X_train = X_train.drop(columns=drop_cols)
X_test = X_test.drop(columns=drop_cols)
print("Remaining sensors:", X_train.shape[1])

Dropping (constant): 122
Remaining sensors: 444


In [5]:
medians = X_train.median()
X_train = X_train.fillna(medians)
X_test = X_test.fillna(medians)          # use TRAIN medians on test
print("Missing left:", X_train.isna().sum().sum(), X_test.isna().sum().sum())

Missing left: 0 0


In [6]:
lo = X_train.quantile(0.005)
hi = X_train.quantile(0.995)

# Record how often test wafers fall outside the training range (for SPC story)
out_of_range = ((X_test < lo) | (X_test > hi)).sum().sort_values(ascending=False)
print("Test readings outside train range (top 10):")
print(out_of_range.head(10))

X_train = X_train.clip(lo, hi, axis=1)
X_test = X_test.clip(lo, hi, axis=1)

Test readings outside train range (top 10):
s6      95
s132    68
s43     62
s143    50
s571    50
s3      45
s82     40
s416    40
s562    39
s67     39
dtype: int64


In [7]:
corr = X_train.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_corr = [c for c in upper.columns if (upper[c] > 0.95).any()]
print("Dropping (corr > 0.95):", len(drop_corr))

X_train = X_train.drop(columns=drop_corr)
X_test = X_test.drop(columns=drop_corr)
print("Final sensor count:", X_train.shape[1])

Dropping (corr > 0.95): 168
Final sensor count: 276


In [8]:
import os
os.makedirs("../data/processed", exist_ok=True)

X_train.assign(fail=y_train.values, timestamp=ts_train.values).to_csv(
    "../data/processed/train.csv", index=False)
X_test.assign(fail=y_test.values, timestamp=ts_test.values).to_csv(
    "../data/processed/test.csv", index=False)
print("Saved.")

Saved.
